In [ ]:
import numpy as np

In [ ]:
import torch 
import torch.nn as nn
import torch.optim as optim 
from torchvision import datasets,transforms
from torch.utils.data import DataLoader
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device}")
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,),(0.5,) )
])


train_data = datasets.MNIST('./data',train=True,download=True,transform=transform)
test_data = datasets.MNIST('./data', train=False,download=True,transform=transform)
train_loader = DataLoader(train_data,batch_size=64,shuffle=True)
test_loader = DataLoader(test_data,batch_size=64,shuffle=False)

INPUT_SIZE   = 28     # pixels per row
HIDDEN_SIZE  = 128    # memory size
NUM_LAYERS   = 2      # stacked LSTMs
NUM_CLASSES  = 10     # digits 0-9
EPOCHS       = 15

class LSTM_(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size = INPUT_SIZE,
            hidden_size = HIDDEN_SIZE,
            num_layers = NUM_LAYERS,
            batch_first=True,
            dropout = 0.3
        
        )
        self.fc = nn.Linear(HIDDEN_SIZE, NUM_CLASSES)
    def forward(self,x ):
        x = x.squeeze(1)
        h0 = torch.zeros(NUM_LAYERS, x.size(0),HIDDEN_SIZE).to(x.device)
        c0 = torch.zeros(NUM_LAYERS, x.size(0), HIDDEN_SIZE).to(x.device)
        out, _ = self.lstm(x(h0,c0))
        out = out[:, -1,: ]
        return out

model = LSTM_().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print("\n " + "=" * 55)
print(f"{'Epoch':<8} {'Loss':>10} {'Accuracy':>12}")
print("="*55)

for epoch in range(EPOCHS):
    model.train()
    total_loss = correct = total = 0
    for images,labels in train_loader:
        images,labels = images.to(device),labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        predicted = torch.argmax(outputs,dim=1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
    avg_loss = total_loss/ len(train_loader)
    accuracy = correct / total * 100
    print(f"{epoch+1:<8} {avg_loss:>10.4f} {accuracy:>11.2f}%")

print("="* 55)
model.eval()
correct = total = 0
with torch.no_grad():
    for images,labels in test_loader:
        images,labels = images.to(device),labels.to(device)
        predicted = torch.argmax(model(images), dim=1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
test_acc = correct / total * 100
print(f"/n test accuarcy : {test_acc:.5f}")
torch.save(model.state_dict(),'mist_lstm.pth')
print("saved ")